# Table 6: White Lies and CDE adoption
Recreate the Table 6 layout from four saved variant-count summaries in `script/res/`. No measurement or classification scripts are rerun.

`variant adoption % = 100 × variant domain count / total mechanism domain count`


In [1]:
from pathlib import Path
import math
import pandas as pd
from IPython.display import display

DATE = "20260406"
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "script/res" / f"{DATE}_nsec_cde_wl.csv").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside dnsnsec26 or set REPO_ROOT explicitly.")
RES_DIR = REPO_ROOT / "script/res"
OUTPUT_DIR = REPO_ROOT / "notebook/results"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org", "umbrella": "Umbrella",
          "tranco": "Tranco", "radar": "Radar", "majestic": "Majestic"}
INPUTS = [
    ("Source", "NSEC", f"{DATE}_nsec_cde_wl.csv", "nseccnt"),
    ("WWW", "NSEC", f"{DATE}_www_nsec_cde_wl.csv", "nseccnt"),
    ("Source", "NSEC3", f"{DATE}_nsec3_cde_wl_everything.csv", "nsec3cnt"),
    ("WWW", "NSEC3", f"{DATE}_nsec3_cde_wl_www_everything.csv", "nsec3cnt"),
]

## Load and validate all four inputs
Each CSV contains one row per dataset, with domain counts for White Lies and CDE. Validate complete dataset coverage, integer counts, positive denominators, and agreement between saved fractions and recomputed fractions.

In [2]:
frames = []
for population, mechanism, filename, total_column in INPUTS:
    frame = pd.read_csv(RES_DIR / filename)
    required = {"zone", total_column, "whitelies", "cde", "wl_perc", "cde_perc"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{filename}: missing columns {required - set(frame.columns)}")
    if frame["zone"].duplicated().any() or set(frame["zone"]) != set(ZONES):
        raise ValueError(f"{filename}: expected one row for every dataset.")
    frame = frame.rename(columns={total_column: "total_domains", "whitelies": "white_lies_count",
                                  "cde": "cde_count", "wl_perc": "saved_wl_fraction",
                                  "cde_perc": "saved_cde_fraction"})
    for column in ["total_domains", "white_lies_count", "cde_count"]:
        values = pd.to_numeric(frame[column], errors="raise")
        if values.isna().any() or values.lt(0).any() or (values % 1 != 0).any():
            raise ValueError(f"{filename}: {column} must contain nonnegative integers.")
        frame[column] = values.astype("int64")
    if (frame["total_domains"].le(0).any()
            or (frame["white_lies_count"] + frame["cde_count"]).gt(frame["total_domains"]).any()):
        raise ValueError(f"{filename}: invalid variant counts or denominators.")
    frame["white_lies_pct"] = 100 * frame["white_lies_count"] / frame["total_domains"]
    frame["cde_pct"] = 100 * frame["cde_count"] / frame["total_domains"]
    for count_column, saved_column in [("white_lies_count", "saved_wl_fraction"),
                                       ("cde_count", "saved_cde_fraction")]:
        saved = pd.to_numeric(frame[saved_column], errors="raise")
        computed = frame[count_column] / frame["total_domains"]
        if not all(math.isclose(a, b, rel_tol=1e-12, abs_tol=1e-15) for a, b in zip(saved, computed)):
            raise ValueError(f"{filename}: saved fractions differ from count / total.")
    frame["population"] = population
    frame["mechanism"] = mechanism
    frame["input_file"] = filename
    frames.append(frame)
stats = pd.concat(frames, ignore_index=True)
print("All four files validated; saved fractions agree with count / total.")

All four files validated; saved fractions agree with count / total.


## Calculated Table 6
The layout is Source NSEC/NSEC3 followed by WWW NSEC/NSEC3, with White Lies and CDE for each mechanism. The supplied paper's second WWW mechanism heading appears as NSEC; context identifies it as NSEC3, which is the heading used here.

Percentages are formatted to two decimals. A displayed 0.00% does not necessarily mean zero domains: source Umbrella NSEC3 has one CDE domain (0.01%), and WWW .com NSEC3 has one CDE domain whose percentage rounds to 0.00%.

In [3]:
paper_columns = pd.MultiIndex.from_tuples([
    (population, mechanism, variant)
    for population in ("Source", "WWW")
    for mechanism in ("NSEC", "NSEC3")
    for variant in ("White Lies", "CDE")
], names=["Population", "Mechanism", "Variant"])
rates = stats.melt(id_vars=["population", "zone", "mechanism"],
                   value_vars=["white_lies_pct", "cde_pct"],
                   var_name="variant", value_name="percentage")
rates["variant"] = rates["variant"].map({"white_lies_pct": "White Lies", "cde_pct": "CDE"})
table6 = rates.pivot(index="zone", columns=["population", "mechanism", "variant"], values="percentage")
table6 = table6.reindex(index=ZONES, columns=paper_columns)
if table6.isna().any().any():
    raise ValueError("Expected all 56 Table 6 percentages.")
table6.index = table6.index.map(LABELS)
table6.index.name = "Dataset"
formatted_table6 = table6.apply(lambda column: column.map(lambda value: f"{value:.2f}%"))
display(formatted_table6)

Population     Source                                  WWW                     \
Mechanism        NSEC              NSEC3              NSEC              NSEC3   
Variant    White Lies     CDE White Lies    CDE White Lies     CDE White Lies   
Dataset                                                                         
.com            0.74%  43.96%      6.98%  0.00%      0.78%  41.92%     16.88%   
.net            0.96%  56.54%      6.86%  0.00%      0.98%  57.47%     13.21%   
.org            0.80%  52.75%      5.81%  0.00%      0.83%  53.75%     15.91%   
Umbrella        1.73%  77.27%      8.11%  0.01%      1.84%  80.14%     10.11%   
Tranco          1.32%  74.95%      9.31%  0.00%      1.37%  77.16%     13.90%   
Radar           0.99%  79.23%     10.37%  0.00%      0.96%  80.32%     14.31%   
Majestic        1.56%  73.11%     11.19%  0.00%      1.52%  73.23%     15.82%   

Population         
Mechanism          
Variant       CDE  
Dataset            
.com        0.00%  
.net        0.00%  
.org        0.00%  
Umbrella    0.00%  
Tranco      0.00%  
Radar       0.00%  
Majestic    0.00%

## Export
- `table6_stats.csv`: counts, input-file provenance, file denominators, and unrounded percentages.
- `table6_formatted.csv`: calculated Table 6 with three header rows.
- `table6_paper_differences.csv`: calculated minus published percentages, in percentage points.
- `table6_mismatches.csv`: the cells that differ from the paper.
- `table6_denominator_audit.csv`: file denominators compared with Table 1 totals.

Rerunning this cell replaces these exports.

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stats.to_csv(OUTPUT_DIR / "table6_stats.csv", index=False)
print(f"Saved Table 6 results to {OUTPUT_DIR}")

Saved Table 6 results to /u/yz6me/dnsnsec26/notebook/results
